# AML - SAIGE-GENE SET-BASED - ASSOCIATION ANALYSIS
This script performs rare variant set-based gene association tests for each chromosome using SAIGE-GENE+. It is configured to use the sparse GRM with 2,000 randomly selected markers to estimate the variance ratio. Maximum minor allele frequency thresholds for each test are set at 1%, 0.1%, and 0.001%. The script evaluates the annotation groups Missense, Loss-of-Function, and a combination of Missense and Loss-of-Function. An additional analysis is conducted using only synonymous variants; this is executed with a separate command so that the Cauchy values from the first test reflect only predicted deleterious variants (Missense and Loss-of-Function).

In [1]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


'gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe'

# COMMAND 

In [8]:
%%writefile high_confidence_analysis.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in biosample_age; do 
    echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
    
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_high_confidence_chr${chrom}" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_synonymous_chr${chrom}" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=Synonymous \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    done
done


Overwriting high_confidence_analysis.sh


In [9]:
!gsutil cp high_confidence_analysis.sh {my_bucket}/data/scripts/

Copying file://high_confidence_analysis.sh [Content-Type=text/x-sh]...
/ [1 files][  2.4 KiB/  2.4 KiB]                                                
Operation completed over 1 objects/2.4 KiB.                                      


In [10]:
!gsutil cat {my_bucket}/data/scripts/high_confidence_analysis.sh

#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    for pheno in biosample_age; do 
    echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
    
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_high_confidence_chr${chrom}" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile=

# SUMBIT COMMAND 

In [11]:
import subprocess
import pandas as pd

chromosome = []
sizes = [] 

# Loop for all chromosome 
for chrom in range(1, 23):
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gsutil",
        "-u", os.getenv("GOOGLE_PROJECT"),
        "ls",
        "-lha",
        f"gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{chrom}.bed"
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

print(genomic_size)

    chromosome    size
0            1  402.73
1            2  286.95
2            3  237.58
3            4  166.42
4            5  192.88
5            6  203.61
6            7  190.62
7            8  151.57
8            9  162.76
9           10  160.09
10          11  235.15
11          12  217.31
12          13   79.48
13          14  135.11
14          15  145.37
15          16  180.08
16          17  219.88
17          18   73.78
18          19   254.5
19          20     100
20          21   40.21
21          22   85.17


In [13]:

# Environment variables 
WORKSPACE_BUCKET = os.getenv("WORKSPACE_BUCKET")
ARTIFACT_REGISTRY_DOCKER_REPO = os.getenv("ARTIFACT_REGISTRY_DOCKER_REPO")

# Logging directory
LOGGING=f'{WORKSPACE_BUCKET}/data/logging'

# Genetic relationship matrix from SAIGE-GENE+ 
GRM=f'{WORKSPACE_BUCKET}/data/saige_step00/grm/'
# Null model from SAIGE-GENE+ 
NULL_GLMM=f'{WORKSPACE_BUCKET}/data/saige_step01/null_glmm/'
# Annotation table 
VAT=f'{WORKSPACE_BUCKET}/data/annotation_table/'

# Output directory 
OUTPUT=f'{WORKSPACE_BUCKET}/data/saige_step02/biosample_collection_check/'
# Dsub script 
SCRIPT=f'{WORKSPACE_BUCKET}/data/scripts/high_confidence_analysis.sh'
# Path to genomic data 
MNT_PATH='/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed'

MAX_DISK_SIZE = 500 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 30 # Set additional disk size for each chromosome 

# Loop to submit dsub command 
for CHROM in range(1,22):
    
    print(f"Submitting dsub command for chromosome {CHROM}")
    
    DISK_SIZE = round(float(genomic_size.loc[genomic_size['chromosome'] == CHROM, 'size'].iloc[0])) + EXTRA_DISK_SIZE
    
    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE: # Set maximum disk size 
        print(f"Disk too large for chromosome {CHROM}")
        break
        
    if DISK_SIZE > 1000: # Absolute maximum for version 8  
        print(f"Disk too large for chromosome {CHROM}")
        break

    BED=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.bed'
    BIM=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.bim'
    FAM=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.fam'

    command = [
        "aou_dsub",
        "--image", f"{ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2",
        "--boot-disk-size", "10",
        "--disk-size", str(DISK_SIZE),
        "--min-ram", "4",
        "--logging", LOGGING,
        "--input", f"plink_bed={BED}",
        "--input", f"plink_bim={BIM}",
        "--input", f"plink_fam={FAM}",
        "--input-recursive", f"grm={GRM}",
        "--input-recursive", f"null_glmm={NULL_GLMM}",
        "--input-recursive", f"vat={VAT}",
        "--output-recursive", f"output={OUTPUT}",
        "--env", f"genos={MNT_PATH}",
        "--env", f"chrom={CHROM}",
        "--script", SCRIPT
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 1
Requesting disk size: 433
high-confi--vienose--260415-130749-08

Job properties:
  job-id: high-confi--vienose--260415-130749-08
  job-name: high-confidence-analysis
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/high-confi--vienose--260415-130749-08-0-0
Launched job-id: high-confi--vienose--260415-130749-08
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'high-confi--vienose--260415-130749-08' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'high-confi--vienose--260415-130749-08' --users 'vienose'

Submitting dsub command for chromosome 2
Requesting disk size: 317
high-confi--vienose--260415-130750-85

Job properties:
  job-id: high-confi--vienose--260415-130750-85
  job-name: high-confidence-analysis
  user-id: 

high-confi--vienose--260415-130808-08

Job properties:
  job-id: high-confi--vienose--260415-130808-08
  job-name: high-confidence-analysis
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc4465fd/locations/us-central1/jobs/high-confi--vienose--260415-130808-08-0-0
Launched job-id: high-confi--vienose--260415-130808-08
To check the status, run:
  dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'high-confi--vienose--260415-130808-08' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --jobs 'high-confi--vienose--260415-130808-08' --users 'vienose'

Submitting dsub command for chromosome 13
Requesting disk size: 109
high-confi--vienose--260415-130809-82

Job properties:
  job-id: high-confi--vienose--260415-130809-82
  job-name: high-confidence-analysis
  user-id: vienose
Provider internal-id (operation): projects/terra-vpc-sc-fc

In [3]:
!dstat --provider google-batch --project terra-vpc-sc-fc4465fd --location us-central1 --users 'vienose' --status '*'

Job Name         Status    Last Update
---------------  --------  --------------
high-confide...  CANCELED  04-15 13:12:34
high-confide...  SUCCESS   04-15 15:05:03
high-confide...  SUCCESS   04-15 17:22:05
high-confide...  SUCCESS   04-15 14:26:43
high-confide...  SUCCESS   04-15 16:42:09
high-confide...  SUCCESS   04-15 16:03:07
high-confide...  SUCCESS   04-15 15:38:24
high-confide...  SUCCESS   04-15 15:22:40
high-confide...  SUCCESS   04-15 14:33:43
high-confide...  SUCCESS   04-15 16:26:35
high-confide...  SUCCESS   04-15 16:45:32
high-confide...  SUCCESS   04-15 15:42:33
high-confide...  SUCCESS   04-15 15:44:39
high-confide...  SUCCESS   04-15 15:35:57
high-confide...  SUCCESS   04-15 16:08:54
high-confide...  SUCCESS   04-15 16:15:52
high-confide...  SUCCESS   04-15 16:05:20
high-confide...  SUCCESS   04-15 15:46:19
high-confide...  SUCCESS   04-15 16:35:24
high-confide...  SUCCESS   04-15 17:01:52
high-confide...  RUNNING   04-15 13:10:21
high-confide...  SUCCESS   04-15 14:5

## DOWNLOAD RESULTS FROM THE WORKSPACE BUCKET 

In [19]:
!rm -rf saige_gene_step02
!rm -rf merged_saige_gene_step02

In [20]:
!mkdir -p saige_gene_step02
!mkdir -p merged_saige_gene_step02

In [4]:
!gsutil cp gs://fc-secure-bf507049-90b6-43f8-86ea-c3ad9e823ffe/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_high_confidence_chr{1..22} saige_gene_step02

CommandException: Destination URL must name a directory, bucket, or bucket
subdirectory for the multiple source form of the cp command.


In [ ]:
%%bash 

# ========================================================================================
# Format gene results for download  
# ========================================================================================

mkdir -p redacted_saige_gene_step02
unmerge_files=saige_gene_step02
redacted_files=redacted_saige_gene_step02

for pheno in aml_phenotype_high_confidence aml_phenotype_synonymous; do 
    for chrom in {1..22}; do
        echo "Processing phenotype: ${pheno} | chr: ${chrom}"
        # Redact each chromosome file 
        awk -v chrom=$chrom 'BEGIN {OFS="\t"} 
            NR==1 {print "Chromosome", $1, $2, $3, $4, $5, $6, $7; next} 
            {print chrom, $1, $2, $3, $4, $5, $6, $7}' \
            $unmerge_files/EUR_${pheno}_chr${chrom} > \
            $redacted_files/EUR_${pheno}_redacted_chr${chrom}
    done
done


In [ ]:
%%bash 

# ========================================================================================
# Format gene results for plotting 
# ========================================================================================

mkdir -p merged_saige_gene_step02
unmerge_files=saige_gene_step02
merge_files=merged_saige_gene_step02

for pheno in aml_phenotype_high_confidence aml_phenotype_synonymous; do 
    echo "Processing phenotype: ${pheno} | chr: 1"
    awk 'BEGIN {OFS="\t"} 
         NR==1 {print $0, "Chromosome"; next} 
        {print $0, 1}' $unmerge_files/EUR_${pheno}_chr1 > $merge_files/EUR_${pheno}_merged_chr
    for chrom in {2..22}; do  # Start from chr2 since chr1 is already used above
         echo "Processing phenotype: ${pheno} | chr: ${chrom}"
         awk -v chrom=$chrom 'BEGIN {OFS="\t"} 
                              NR==1 {print $0, "Chromosome"; next} 
                              {print $0, chrom}' $unmerge_files/EUR_${pheno}_chr${chrom} > $unmerge_files/EUR_${pheno}_formatted_chr${chrom}
        tail +2 $unmerge_files/EUR_${pheno}_formatted_chr${chrom} >> $merge_files/EUR_${pheno}_merged_chr
    done
done


In [4]:
import glob
import os
import pandas as pd 
import mygene
import re

# ========================================================================================
# Merge together results across chromosomes 
# ========================================================================================
files = []
for chrom in range(1,23):
    file = pd.read_csv(f'{my_bucket}/data/saige_step02/biosample_collection_check/EUR_biosample_age_high_confidence_chr{chrom}',
                      sep = '\t')
    files.append(file)
    
files = pd.concat(files, axis = 0)

# ========================================================================================
# Format table of significant gene results 
# ========================================================================================
non_cauchy_pvalues = files[~(files['Group'] == 'Cauchy')]
non_cauchy_pvalues = non_cauchy_pvalues[['Region', 'Group', 'max_MAF', 
                                         'Pvalue', 'Pvalue_Burden', 'Pvalue_SKAT', 
                                         'BETA_Burden', 'SE_Burden']]

non_cauchy_pvalues['Bonferroni SKAT-O P-value'] = (
    (non_cauchy_pvalues['Pvalue'] * len(non_cauchy_pvalues))
)

non_cauchy_pvalues = (
    non_cauchy_pvalues[non_cauchy_pvalues['Pvalue'] < (0.05/len(non_cauchy_pvalues))]
                       )

mg = mygene.MyGeneInfo()

result = mg.querymany(non_cauchy_pvalues['Region'], 
                      scopes='ensembl.gene', 
                      fields='symbol', 
                      species='human')
symbols = []
for r in result: 
     symbols.append(r.get('symbol'))
symbols = pd.DataFrame(symbols, columns=["Gene Symbol"])

non_cauchy_pvalues['Gene Symbol'] = symbols['Gene Symbol'].values

non_cauchy_pvalues['Group'] = (
    non_cauchy_pvalues['Group'].str.replace(r';', ' & ', regex=True)
)

non_cauchy_pvalues = non_cauchy_pvalues.rename(columns={
    'max_MAF': 'Maximum MAF',
    'Pvalue': 'SKAT-O P-value',
    'Pvalue_Burden': 'Burden P-value',
    'Pvalue_SKAT': 'SKAT P-value',
    'BETA_Burden': 'Burden Beta',
    'SE_Burden': 'Burden Standard Error',
    'Group': 'Annotation Group'
})

non_cauchy_pvalues = (
    non_cauchy_pvalues[['Gene Symbol', 'Annotation Group', 'Maximum MAF', 
       'SKAT-O P-value', 'Bonferroni SKAT-O P-value','Burden P-value', 'SKAT P-value', 
       'Burden Beta','Burden Standard Error']]
)
    
# ========================================================================================
# Format table of significant cauchy gene results 
# ========================================================================================
cauchy_pvalues = files[(files['Group'] == 'Cauchy')]

cauchy_pvalues['Bonferroni Cauchy SKAT-O P-value'] = (
    (cauchy_pvalues['Pvalue'] * len(cauchy_pvalues))
)

cauchy_pvalues = (
    cauchy_pvalues[cauchy_pvalues['Pvalue'] < (0.05/len(cauchy_pvalues))]
                       )

mg = mygene.MyGeneInfo()

result = mg.querymany(cauchy_pvalues['Region'], 
                      scopes='ensembl.gene', 
                      fields='symbol', 
                      species='human')
symbols = []
for r in result: 
     symbols.append(r.get('symbol'))
symbols = pd.DataFrame(symbols, columns=["Gene Symbol"])

cauchy_pvalues['Gene Symbol'] = symbols['Gene Symbol'].values


cauchy_pvalues = cauchy_pvalues.rename(columns={
    'max_MAF': 'Maximum MAF',
    'Pvalue': 'Cauchy SKAT-O P-value',
    'Pvalue_Burden': 'Cauchy Burden P-value',
    'Pvalue_SKAT': 'Cauchy SKAT P-value',
    'BETA_Burden': 'Burden Beta',
    'SE_Burden': 'Burden Standard Error',
    'Group': 'Annotation Group'
})

cauchy_pvalues = (
    cauchy_pvalues[['Gene Symbol',
       'Cauchy SKAT-O P-value', 
       'Bonferroni Cauchy SKAT-O P-value',
       'Cauchy Burden P-value', 
       'Cauchy SKAT P-value']]
)

# ========================================================================================
# Save significant gene results to excel files 
# ========================================================================================
non_cauchy_pvalues.to_excel("eur_aml_phenotype_non_cauchy_pvalues.xlsx", index=False)
non_cauchy_pvalues.to_csv("eur_aml_phenotype_non_cauchy_pvalues.tsv", 
                          sep ='\t', index=False)
cauchy_pvalues.to_excel("eur_aml_phenotype_cauchy_pvalues.xlsx", index=False)
cauchy_pvalues.to_csv("eur_aml_phenotype_cauchy_pvalues.tsv", 
                          sep ='\t', index=False)

10 input query terms found dup hits:	[('ENSG00000078369', 4), ('ENSG00000115524', 6), ('ENSG00000119772', 9), ('ENSG00000168769', 9), ('E
/tmp/ipykernel_1126/1995122364.py:72: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cauchy_pvalues['Bonferroni Cauchy SKAT-O P-value'] = (


In [5]:
cauchy_pvalues

,Gene Symbol,Cauchy SKAT-O P-value,Bonferroni Cauchy SKAT-O P-value,Cauchy Burden P-value,Cauchy SKAT P-value
1382,GNB1,2.473693e-19,4.468478e-15,6.061818e-14,3.862076e-20
2281,SF3B1,2.627076e-91,4.745551e-87,3.752966e-92,1.906714e-55
3359,DNMT3A,1.256862e-293,2.270395e-289,0.000000e+00,2.394668e-254
4772,TET2,1.671418e-229,3.019250e-225,4.340458e-225,4.078747e-213
4037,MTA2,4.721190e-10,8.528358e-06,4.729004e-10,4.606685e-04
4364,TP53,9.769963e-15,1.764846e-10,3.330669e-15,1.776357e-15
5222,SRSF2,6.531213e-72,1.179798e-67,1.065336e-67,1.251871e-56
6465,PPM1D,2.196069e-60,3.966979e-56,3.137241e-61,1.203200e-48
3997,ASXL1,3.939028e-35,7.115460e-31,5.627183e-36,1.919969e-28
2894,CHEK2,8.740557e-10,1.578894e-05,1.229145e-08,6.137855e-10
